# Robo Flow — Colab auto-label (1–3 hour session)

Label images on **Google Colab GPU**. If Colab fails, the script **automatically queues the job on Railway**.

## What you must add in Colab (copy from Railway / Supabase / Hugging Face)

| Name | Where to find it | Example |
|------|------------------|---------|
| `SUPABASE_URL` | Supabase → Settings → API | `https://xxx.supabase.co` |
| `SUPABASE_SERVICE_ROLE_KEY` | Supabase → service_role key | `eyJ...` |
| `HF_TOKEN` | huggingface.co → Settings → Access Tokens | `hf_...` |
| `HF_DATASET_REPO` | Railway env (images repo) | `Adeel6480/robo_flow` |
| `HF_MODEL_REPO` | Railway env (models repo) | `Adeel6480/robo_flow` |
| `RAILWAY_WORKER_URL` | Railway public URL | `https://roboflow-production.up.railway.app` |
| `WORKER_API_KEY` | Railway env (same as Vercel) | only if you set it on Railway |

## What you must set manually (from your app)

| Variable | Where |
|----------|--------|
| `PROJECT_ID` | Project URL or Supabase `projects` table |
| `DATASET_ID` | Dataset page URL |
| `MODEL_IDS` | Models page — UUIDs, comma-separated |

**Before running:** images on HF, classes added, models uploaded.

**Runtime:** menu → **Change runtime type** → **T4 GPU**

## 1) Clone repo & install

In [ ]:
REPO_URL = "https://github.com/adeeltariq6480/robo_flow.git"

!git clone {REPO_URL} robo_flow 2>/dev/null || (cd robo_flow && git pull)
%cd robo_flow/worker

!pip install -q -r requirements.txt

## 2) Secrets (Colab: Runtime → secrets, or paste below)

In [ ]:
import os

# --- PASTE YOUR VALUES (or use Colab Secrets: key icon on the left) ---
os.environ["SUPABASE_URL"] = "https://YOUR_PROJECT.supabase.co"
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = "YOUR_SERVICE_ROLE_KEY"
os.environ["HF_TOKEN"] = "hf_..."
os.environ["HF_DATASET_REPO"] = "YourUser/robo_flow"
os.environ["HF_MODEL_REPO"] = "YourUser/robo_flow"
os.environ["HF_DATASET_REPO_TYPE"] = "dataset"
os.environ["HF_MODEL_REPO_TYPE"] = "model"

# Railway fallback (auto-used if Colab fails)
os.environ["RAILWAY_WORKER_URL"] = "https://roboflow-production.up.railway.app"
os.environ["WORKER_API_KEY"] = ""  # leave empty if Railway has no API key

# Optional: load from Colab Secrets (same names as table above)
try:
    from google.colab import userdata
    for key in (
        "SUPABASE_URL",
        "SUPABASE_SERVICE_ROLE_KEY",
        "HF_TOKEN",
        "HF_DATASET_REPO",
        "HF_MODEL_REPO",
        "RAILWAY_WORKER_URL",
        "WORKER_API_KEY",
    ):
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    pass

## 3) Your project IDs (from app URL or Supabase)

In [ ]:
PROJECT_ID = "your-project-uuid"
DATASET_ID = "your-dataset-uuid"
# Comma-separated model UUIDs from Models page
MODEL_IDS = "model-uuid-1,model-uuid-2"

CONFIDENCE = 0.15
IOU = 0.45
# False = only unlabeled | True = only already labeled
RELABEL = False

## 4) Run labeling

- Tries **Colab GPU** first
- If Colab errors / disconnects → **auto-fallback to Railway**
- Keep tab open 1–3 hours (or watch job in Vercel app)

In [ ]:
relabel_flag = "--relabel" if RELABEL else ""

!python scripts/colab_auto_label.py \
  --project-id {PROJECT_ID} \
  --dataset-id {DATASET_ID} \
  --model-ids {MODEL_IDS} \
  --confidence {CONFIDENCE} \
  --iou {IOU} \
  {relabel_flag}

## 5) After finish

1. Open **Vercel app** → Inference / Label page (job progress) or Review
2. Labels in **Supabase**; `.txt` files on **Hugging Face** after job completes
3. If you saw `Auto-fallback → Railway`, Colab failed but labeling continued on Railway

### Force Railway only (skip Colab)
Add flag: `--railway-only` to the command in cell 4